# Consolidacao ~500k Final — Fase A.1 / passo 2

**Spec**: `specs/004-melhoria-geracao-dados-cnn/PRD.md` §4.1.3 rev.3 (2026-05-08).

Combina tres fontes num conjunto final respeitando cotas por celula `(gen_mode, bucket)`:

| Fonte | Caminho | Estados brutos | Unicos |
|---|---|---:|---:|
| Legado | `dados/profundidade_minmax_9/` | 344.000 | 226.201 |
| V5_Databricks | `dados/profundidade_minmax_9_v5_databricks/` | 185.000 | 139.903 |
| V5_Local | `dados/profundidade_minmax_9_v5_local/` | ~350k | ~220k |

**Distribuicao-alvo final (2026-05-08 rev.3 — (24,28) e (29,30) capeados; cota de (24,28) redistribuida):**

| Bucket | Amostras | % | Observacao |
|---|---:|---:|---|
| 5–11 | 55.501 | 11,10% | Inalterado |
| 12–17 | 157.588 | 31,52% | +46.587 redistribuidos de (24,28) |
| 18–23 | 220.623 | 44,12% | +65.222 redistribuidos de (24,28) |
| 24–28 | 65.792 | 13,16% | CAPEADO — autoplay sim_l2/sim_l3 satura em ~57.020 pos. unicas; C(31,24..28)=991.333 teorico |
| **29–30** | **496** | **0,10%** | LIMITE FISICO: C(31,29)+C(31,30)=465+31=496 |
| **Total** | **500.000** | | |

**Shortfall esperado:** ate ~6.000 estados abaixo do alvo, por duas restricoes de distribuicao de modos:
- **(29–30)**: ~295 (mode_2 tem 80 disponiveis mas cota=198; mode_3 tem 96 mas cota=273).
- **(24–28)**: ate ~5.500 — autoplay (sim_l2/sim_l3) satura em ~57.020 pos. unicas,
  mas a cota de modes 2+3 combinada é ~62.500. Mode_0 tem excedente descartado.
  Diferenca total maxima < 1,2% do alvo, irrelevante para treinamento da CNN.

**Regras de consolidacao:**
1. **Mix de `gen_mode`** (D1.a): pesos 5/40/55% para uniform/sim_l2/sim_l3. `sim_l1` (modo 1) descartado.
2. **Cota por celula** `cota_alvo[m,b] = round(BUCKET_ALVO[b] * PESO_GEN[m])`.
3. **Dedup global** por `mat.tobytes()`.
4. Ordem de prioridade: legado → v5_databricks → v5_local.

**Saida**: `dados/profundidade_minmax_9_final/dataset_pequeno_*.npz` em lotes de 5.000.
Pronto para `Enriquece_NPZ_Com_Canais.ipynb`.

In [20]:
import os, glob, json, time
import random as _random
from collections import Counter, defaultdict
import numpy as np

REPO_ROOT        = r'D:\Desenvolvimento\arena-sagaz\arena-sagaz-backend'
DIR_LEGADO       = os.path.join(REPO_ROOT, 'dados', 'profundidade_minmax_9_desbalanceado')
DIR_V5_DATABRICKS= os.path.join(REPO_ROOT, 'dados', 'profundidade_minmax_9_v5_databricks')
DIR_V5_LOCAL     = os.path.join(REPO_ROOT, 'dados', 'profundidade_minmax_9_v5_local')
DIR_FINAL        = os.path.join(REPO_ROOT, 'dados', 'profundidade_minmax_9')
os.makedirs(DIR_FINAL, exist_ok=True)

TAMANHO_LOTE = 5_000
SEED_GLOBAL  = 42

print(f'DIR_LEGADO        : {DIR_LEGADO}')
print(f'DIR_V5_DATABRICKS : {DIR_V5_DATABRICKS}')
print(f'DIR_V5_LOCAL      : {DIR_V5_LOCAL}')
print(f'DIR_FINAL         : {DIR_FINAL}')
print(f'TAMANHO_LOTE: {TAMANHO_LOTE:,}')


DIR_LEGADO        : D:\Desenvolvimento\arena-sagaz\arena-sagaz-backend\dados\profundidade_minmax_9_desbalanceado
DIR_V5_DATABRICKS : D:\Desenvolvimento\arena-sagaz\arena-sagaz-backend\dados\profundidade_minmax_9_v5_databricks
DIR_V5_LOCAL      : D:\Desenvolvimento\arena-sagaz\arena-sagaz-backend\dados\profundidade_minmax_9_v5_local
DIR_FINAL         : D:\Desenvolvimento\arena-sagaz\arena-sagaz-backend\dados\profundidade_minmax_9
TAMANHO_LOTE: 5,000


In [21]:
# === Cota alvo — PRD §4.1.3 rev.5 (2026-05-08) ===
#
# rev.5: todas as celulas capeadas nos unicos reais disponiveis.
#   Excedente redistribuido para mode_3 em (12-17) e (18-23).
#
# Celulas capeadas (rev.5):
#   mode_2 (12,17): 67.898 unicos disponiveis (cota rev.4: 68.597, delta -699)
#   mode_2 (18,23): 83.787 unicos disponiveis (cota rev.4: 97.705, delta -13.918)
#   mode_2 (29,30): 84 unicos (cota rev.4: 87, delta -3)
#   mode_3 (24,28): 21.261 unicos (cota rev.4: 21.458, delta -197)
#   Total liberado: 14.817
#
# Redistribuicao para mode_3 (tem excedente de unicos):
#   mode_3 (18,23): 121.343 + 7.392 = 128.735 (cap no max disponivel: 128.735)
#   mode_3 (12,17): 86.674 + 7.425 = 94.099 (disponivel: 104.010 >> 94.099)
#   Total redistribuido: 7.392 + 7.425 = 14.817 ✓
#
# Mix gen_mode: mode_0=5.0%, mode_2=40.1%, mode_3=54.9%
#   (praticamente inalterado do alvo 5/40/55).

PESO_GEN    = {0: 0.05, 2: 0.40, 3: 0.55}
BUCKETS     = [(5, 11), (12, 17), (18, 23), (24, 28), (29, 30)]
ALVO_TOTAL  = 500_000

# Cotas explicitas — capeadas nos unicos reais de todas as fontes.
cota_alvo = {
    # mode_0: inalterado
    (0, (5, 11)):  2_775,
    (0, (12, 17)): 7_879,
    (0, (18, 23)): 11_031,
    (0, (24, 28)): 3_289,
    (0, (29, 30)):    25,
    # mode_2: (12,17) e (18,23) capeados nos unicos reais
    (2, (5, 11)):  22_200,
    (2, (12, 17)): 67_898,   # cap: 67.898 unicos disponiveis
    (2, (18, 23)): 83_787,   # cap: 83.787 unicos disponiveis
    (2, (24, 28)): 26_317,
    (2, (29, 30)):     84,   # cap: 84 unicos disponiveis
    # mode_3: (12,17) e (18,23) recebem redistribuicao; (24,28) capeado
    (3, (5, 11)):  30_526,
    (3, (12, 17)): 94_099,   # 86.674 + 7.425 redistribuidos
    (3, (18, 23)): 128_735,  # 121.343 + 7.392 redistribuidos (cap no max disponivel)
    (3, (24, 28)): 21_261,   # cap: 21.261 unicos disponiveis
    (3, (29, 30)):     94,
}

_total = sum(cota_alvo.values())
assert _total == ALVO_TOTAL, f'Soma cota_alvo {_total:,} != {ALVO_TOTAL:,}'

print(f'cota_alvo (total {_total:,}):')
print(f"  {'celula':<22} {'cota':>7}")
for m in sorted(PESO_GEN):
    for b in BUCKETS:
        print(f'  mode_{m} bucket {str(b):<10} {cota_alvo[(m, b)]:>7,}')
    print(f"  {'  -- subtotal mode_' + str(m):<22} {sum(cota_alvo[(m, b)] for b in BUCKETS):>7,}")
print(f'\nNOTA rev.5: todas as cotas capeadas nos unicos reais. Zero shortfall esperado.')


def bucket_de(t):
    return next((b for b in BUCKETS if b[0] <= t <= b[1]), None)

cota_alvo (total 500,000):
  celula                    cota
  mode_0 bucket (5, 11)      2,775
  mode_0 bucket (12, 17)     7,879
  mode_0 bucket (18, 23)    11,031
  mode_0 bucket (24, 28)     3,289
  mode_0 bucket (29, 30)        25
    -- subtotal mode_0    24,999
  mode_2 bucket (5, 11)     22,200
  mode_2 bucket (12, 17)    67,898
  mode_2 bucket (18, 23)    83,787
  mode_2 bucket (24, 28)    26,317
  mode_2 bucket (29, 30)        84
    -- subtotal mode_2   200,286
  mode_3 bucket (5, 11)     30,526
  mode_3 bucket (12, 17)    94,099
  mode_3 bucket (18, 23)   128,735
  mode_3 bucket (24, 28)    21,261
  mode_3 bucket (29, 30)        94
    -- subtotal mode_3   274,715

NOTA rev.5: todas as cotas capeadas nos unicos reais. Zero shortfall esperado.


In [22]:
# === Estado da consolidacao (acumulado entre os passos 1 e 2) ===

hashes_unicos = set()
aceitos = {k: 0 for k in cota_alvo}     # contador por celula (mode, bucket)
estados_out, rotulos_out, scores_out, modes_out = [], [], [], []
labels_canon = None
minimax_depth_canon  = None

# Contadores de descarte (uteis pra auditoria do que veio do legado).
descartado = defaultdict(int)
origem_aceitos = defaultdict(int)        # 'legado' | 'v5_local'


def processa_arquivo(arq: str, origem: str) -> None:
    """Le um NPZ e tenta aceitar cada estado conforme regras de cota + dedup."""
    global labels_canon, minimax_depth_canon
    d = np.load(arq, allow_pickle=True)
    estados = d['estados']
    rotulos = d['rotulos']
    scores  = d['scores']
    modes   = d['generation_mode']

    # Schema: snapshot do primeiro arquivo, valida demais.
    if labels_canon is None:
        labels_canon = np.array(d['labels_canonicos'])
        minimax_depth_canon  = np.array(d['minimax_depth'])
    else:
        if not np.array_equal(labels_canon, d['labels_canonicos']):
            raise ValueError(f'labels_canonicos divergente em {arq}')
        if not np.array_equal(minimax_depth_canon, d['minimax_depth']):
            raise ValueError(f'minimax_depth divergente em {arq}')

    n = len(estados)
    indices = list(range(n))
    rng = _random.Random(hash(os.path.basename(arq)) & 0x7FFFFFFF)
    rng.shuffle(indices)

    for i in indices:
        mat = estados[i]
        m   = int(modes[i])
        if m == 1:
            descartado['sim_l1'] += 1
            continue
        if m not in PESO_GEN:
            descartado['mode_invalido'] += 1
            continue
        n_tracos = int((mat == 9).sum())
        b = bucket_de(n_tracos)
        if b is None:
            descartado['bucket_fora'] += 1
            continue
        h = mat.tobytes()
        if h in hashes_unicos:
            descartado['duplicado'] += 1
            continue
        cell = (m, b)
        if aceitos[cell] >= cota_alvo[cell]:
            descartado['cota_excedida'] += 1
            continue

        hashes_unicos.add(h)
        aceitos[cell] += 1
        origem_aceitos[origem] += 1
        estados_out.append(np.array(mat, dtype=np.int8, copy=True))
        rotulos_out.append(str(rotulos[i]))
        scores_out.append(np.array(scores[i], dtype=np.float32, copy=True))
        modes_out.append(m)


def imprime_aceitos_por_celula(titulo: str) -> None:
    print(f'\n--- {titulo} ---')
    print(f"  {'celula':<22} {'aceitos':>8} {'cota':>8} {'falta':>8}")
    for m in sorted(PESO_GEN):
        for b in BUCKETS:
            c = (m, b)
            print(f'  mode_{m} bucket {str(b):<10} {aceitos[c]:>8,} {cota_alvo[c]:>8,} {cota_alvo[c]-aceitos[c]:>8,}')
    print(f"  {'TOTAL':<22} {sum(aceitos.values()):>8,} {sum(cota_alvo.values()):>8,} {sum(cota_alvo.values())-sum(aceitos.values()):>8,}")


print('Estado inicial pronto.')

Estado inicial pronto.


In [23]:
# === Passo 1: legado ===
# Aceita ate cota_alvo[m,b] de cada celula. O excedente (incluindo modo 1
# inteiro, bucket <5, e excedentes por cota) e descartado.

t0 = time.time()
arqs_leg = sorted(glob.glob(os.path.join(DIR_LEGADO, 'dataset_pequeno_*.npz')))
if not arqs_leg:
    raise FileNotFoundError(f'Nenhum NPZ encontrado em {DIR_LEGADO}')
print(f'Lendo {len(arqs_leg)} arquivos legado...')
for k, arq in enumerate(arqs_leg, 1):
    processa_arquivo(arq, 'legado')
    if k % 10 == 0 or k == len(arqs_leg):
        print(f'  {k}/{len(arqs_leg)} | aceitos={sum(aceitos.values()):,} | t={time.time()-t0:.1f}s')

imprime_aceitos_por_celula('Apos passo 1 (legado)')
print(f"\nDescartado nesta fase (acumulado):")
for k, v in sorted(descartado.items()):
    print(f'  {k}: {v:,}')

Lendo 58 arquivos legado...
  10/58 | aceitos=41,889 | t=0.3s
  20/58 | aceitos=79,701 | t=0.6s
  30/58 | aceitos=108,648 | t=0.8s
  40/58 | aceitos=132,508 | t=1.1s
  50/58 | aceitos=154,156 | t=1.3s
  58/58 | aceitos=168,661 | t=1.5s

--- Apos passo 1 (legado) ---
  celula                  aceitos     cota    falta
  mode_0 bucket (5, 11)       2,775    2,775        0
  mode_0 bucket (12, 17)      7,879    7,879        0
  mode_0 bucket (18, 23)     11,031   11,031        0
  mode_0 bucket (24, 28)      3,289    3,289        0
  mode_0 bucket (29, 30)          0       25       25
  mode_2 bucket (5, 11)      22,200   22,200        0
  mode_2 bucket (12, 17)     48,722   67,898   19,176
  mode_2 bucket (18, 23)     45,348   83,787   38,439
  mode_2 bucket (24, 28)     12,522   26,317   13,795
  mode_2 bucket (29, 30)          0       84       84
  mode_3 bucket (5, 11)       5,038   30,526   25,488
  mode_3 bucket (12, 17)      4,444   94,099   89,655
  mode_3 bucket (18, 23)      4,1

In [24]:
# === Passo 2: v5_databricks + v5_local (complemento) ===
# Prioridade: v5_databricks primeiro (menor sobrefluxo esperado), depois v5_local.

t1 = time.time()
for fonte, diretorio in [('v5_databricks', DIR_V5_DATABRICKS), ('v5_local', DIR_V5_LOCAL)]:
    arqs = sorted(glob.glob(os.path.join(diretorio, 'dataset_pequeno_*.npz')))
    if not arqs:
        print(f'AVISO: nenhum NPZ em {diretorio} ({fonte}).')
        continue
    print(f'Lendo {len(arqs)} arquivos {fonte}...')
    for k, arq in enumerate(arqs, 1):
        processa_arquivo(arq, fonte)
        if k % 10 == 0 or k == len(arqs):
            print(f'  {k}/{len(arqs)} | aceitos={sum(aceitos.values()):,} | t={time.time()-t1:.1f}s')

imprime_aceitos_por_celula('Apos passo 2 (legado + v5_databricks + v5_local)')
print(f"\nDescartado total ate aqui:")
for k, v in sorted(descartado.items()):
    print(f'  {k}: {v:,}')
print(f"\nOrigem dos aceitos:")
for k, v in sorted(origem_aceitos.items()):
    print(f'  {k}: {v:,}')

_total_aceitos = sum(aceitos.values())
_falta = ALVO_TOTAL - _total_aceitos
if _falta == 0:
    print(f'\nTotal {_total_aceitos:,} — 500.000 exato. Pronto para gravar.')
elif _falta <= 1_000:
    print(f'\nTotal {_total_aceitos:,} — shortfall residual de {_falta:,} (< 0,2%). Pronto para gravar.')
else:
    print(f'\nATENCAO: faltam {_falta:,} estados. Gerar mais dados no V5_Local antes de gravar.')

Lendo 37 arquivos v5_databricks...
  10/37 | aceitos=218,156 | t=0.3s
  20/37 | aceitos=267,737 | t=0.6s
  30/37 | aceitos=317,368 | t=1.0s
  37/37 | aceitos=350,117 | t=1.2s
Lendo 50 arquivos v5_local...
  10/50 | aceitos=382,841 | t=1.5s
  20/50 | aceitos=414,738 | t=1.8s
  30/50 | aceitos=444,475 | t=2.0s
  40/50 | aceitos=462,838 | t=2.3s
  50/50 | aceitos=499,997 | t=2.6s

--- Apos passo 2 (legado + v5_databricks + v5_local) ---
  celula                  aceitos     cota    falta
  mode_0 bucket (5, 11)       2,775    2,775        0
  mode_0 bucket (12, 17)      7,879    7,879        0
  mode_0 bucket (18, 23)     11,031   11,031        0
  mode_0 bucket (24, 28)      3,289    3,289        0
  mode_0 bucket (29, 30)         25       25        0
  mode_2 bucket (5, 11)      22,200   22,200        0
  mode_2 bucket (12, 17)     67,897   67,898        1
  mode_2 bucket (18, 23)     83,787   83,787        0
  mode_2 bucket (24, 28)     26,317   26,317        0
  mode_2 bucket (29, 30)

In [25]:
# === Gravacao do consolidado ===
# Embaralha a ordem global (seed fixa) para misturar legado/v5_local entre os NPZs
# de saida. Sem isso os primeiros NPZs seriam so legado e os ultimos so v5_local.

N = len(estados_out)
if N == 0:
    raise RuntimeError('Nada a gravar. Verifique se os diretorios de entrada tem dados.')

indices = list(range(N))
_random.Random(SEED_GLOBAL).shuffle(indices)

# Limpa NPZs anteriores em DIR_FINAL para evitar mistura com rodada antiga.
antigos = sorted(glob.glob(os.path.join(DIR_FINAL, 'dataset_pequeno_*.npz')))
if antigos:
    print(f'Removendo {len(antigos)} NPZs anteriores em {DIR_FINAL}...')
    for arq in antigos:
        os.remove(arq)

ul = 0
t2 = time.time()
for i0 in range(0, N, TAMANHO_LOTE):
    chunk = indices[i0:i0 + TAMANHO_LOTE]
    ul += 1
    path = os.path.join(DIR_FINAL, f'dataset_pequeno_{ul:04d}.npz')
    np.savez_compressed(
        path,
        estados          = np.array([estados_out[i] for i in chunk], dtype=np.int8),
        rotulos          = np.array([rotulos_out[i] for i in chunk], dtype=str),
        scores           = np.array([scores_out[i] for i in chunk], dtype=np.float32),
        generation_mode  = np.array([modes_out[i] for i in chunk], dtype=np.int8),
        labels_canonicos = labels_canon,
        minimax_depth = minimax_depth_canon,
    )
    if ul % 10 == 0 or i0 + TAMANHO_LOTE >= N:
        print(f'  Batch {ul:04d} -> {os.path.basename(path)} ({len(chunk)} estados) | t={time.time()-t2:.1f}s')

print(f'\nGravados {ul} NPZs em {DIR_FINAL} ({N:,} estados, {time.time()-t2:.1f}s).')

  Batch 0010 -> dataset_pequeno_0010.npz (5000 estados) | t=0.3s
  Batch 0020 -> dataset_pequeno_0020.npz (5000 estados) | t=0.5s
  Batch 0030 -> dataset_pequeno_0030.npz (5000 estados) | t=0.8s
  Batch 0040 -> dataset_pequeno_0040.npz (5000 estados) | t=1.1s
  Batch 0050 -> dataset_pequeno_0050.npz (5000 estados) | t=1.3s
  Batch 0060 -> dataset_pequeno_0060.npz (5000 estados) | t=1.6s
  Batch 0070 -> dataset_pequeno_0070.npz (5000 estados) | t=1.9s
  Batch 0080 -> dataset_pequeno_0080.npz (5000 estados) | t=2.1s
  Batch 0090 -> dataset_pequeno_0090.npz (5000 estados) | t=2.4s
  Batch 0100 -> dataset_pequeno_0100.npz (4997 estados) | t=2.7s

Gravados 100 NPZs em D:\Desenvolvimento\arena-sagaz\arena-sagaz-backend\dados\profundidade_minmax_9 (499,997 estados, 2.7s).


In [26]:
# === Auditoria pos-gravacao ===

arqs = sorted(glob.glob(os.path.join(DIR_FINAL, 'dataset_pequeno_*.npz')))
hashes_aud  = set()
modes_aud   = []
tracos_aud  = []
for arq in arqs:
    d = np.load(arq, allow_pickle=True)
    for i in range(len(d['estados'])):
        mat = d['estados'][i]
        hashes_aud.add(mat.tobytes())
        modes_aud.append(int(d['generation_mode'][i]))
        tracos_aud.append(int((mat == 9).sum()))

total = len(modes_aud)
print(f'Total auditado: {total:,}')
print(f'Unicos por mat.tobytes(): {len(hashes_aud):,}')
assert len(hashes_aud) == total, 'Existem duplicatas no consolidado.'

# Distribuicao por bucket (alvo derivado de cota_alvo)
bucket_alvo_aud = {}
for (m, b), v in cota_alvo.items():
    bucket_alvo_aud[b] = bucket_alvo_aud.get(b, 0) + v

cont_buc = Counter(bucket_de(t) for t in tracos_aud)
print('\nDistribuicao por bucket:')
print(f"  {'Bucket':<10} {'N':>8} {'%real':>7} {'alvo_N':>8} {'desvio_pp':>10}")
for b in BUCKETS:
    n = cont_buc.get(b, 0)
    pct = n / total * 100 if total else 0
    alvo_n = bucket_alvo_aud[b]
    alvo_pct = alvo_n / ALVO_TOTAL * 100
    desv = pct - alvo_pct
    tol = 2.0
    flag = 'OK' if abs(desv) <= tol else 'FORA DA TOLERANCIA'
    print(f'  {str(b):<10} {n:>8,} {pct:>6.2f}% {alvo_n:>8,} {desv:>+9.2f}  {flag}')

# Mix de gen_mode
cont_mode = Counter(modes_aud)
print('\nMix de gen_mode:')
for m, p in sorted(PESO_GEN.items()):
    n = cont_mode.get(m, 0)
    pct = n / total * 100 if total else 0
    alvo = p * 100
    desv = pct - alvo
    print(f'  mode_{m}: {n:>7,} ({pct:>5.2f}%, alvo {alvo:.2f}%, desvio {desv:+.2f}pp)')

# sim_l1 zerado
n_sim_l1 = cont_mode.get(1, 0)
print(f'\nsim_l1 (mode 1): {n_sim_l1} (esperado 0)')
assert n_sim_l1 == 0, f'sim_l1 deve estar 0, encontrado {n_sim_l1}'

# Total: aceita shortfall residual minimo (arredondamentos).
# rev.4 eliminou o shortfall de celulas saturadas via redistribuicao.
SHORTFALL_MAX = 1_000
if total >= ALVO_TOTAL - SHORTFALL_MAX:
    print(f'\nAUDITORIA OK — {total:,} consolidados (shortfall {ALVO_TOTAL-total:,}, max aceito {SHORTFALL_MAX:,}).')
    print('Pronto para Enriquece_NPZ_Com_Canais.ipynb.')
else:
    print(f'\nATENCAO: total {total:,} < {ALVO_TOTAL - SHORTFALL_MAX:,} (shortfall {ALVO_TOTAL-total:,} > {SHORTFALL_MAX:,}).')
    print('Verificar se o V5_Local terminou a geracao — rodar COMPLEMENTO_POR_CELULA rev.4.')

Total auditado: 499,997
Unicos por mat.tobytes(): 499,997

Distribuicao por bucket:
  Bucket            N   %real   alvo_N  desvio_pp
  (5, 11)      55,501  11.10%   55,501     +0.00  OK
  (12, 17)    169,875  33.98%  169,876     +0.00  OK
  (18, 23)    223,551  44.71%  223,553     -0.00  OK
  (24, 28)     50,867  10.17%   50,867     +0.00  OK
  (29, 30)        203   0.04%      203     +0.00  OK

Mix de gen_mode:
  mode_0:  24,999 ( 5.00%, alvo 5.00%, desvio -0.00pp)
  mode_2: 200,285 (40.06%, alvo 40.00%, desvio +0.06pp)
  mode_3: 274,713 (54.94%, alvo 55.00%, desvio -0.06pp)

sim_l1 (mode 1): 0 (esperado 0)

AUDITORIA OK — 499,997 consolidados (shortfall 3, max aceito 1,000).
Pronto para Enriquece_NPZ_Com_Canais.ipynb.
